# 📊 Credit Risk Scoring untuk UMKM — EDA & Feature Engineering

**Datathon Project — Ekonomi Digital & Inklusi Keuangan**

## 🎯 Tujuan Notebook
1. Memahami karakteristik dataset credit risk
2. Melakukan data quality audit & cleaning
3. Eksplorasi hubungan antar fitur dengan target (default)
4. Feature engineering yang mengandung business logic
5. Menyiapkan dataset siap-modeling untuk Azure ML

## 🏗️ Konteks Bisnis
Dataset ini akan kami gunakan sebagai **proxy untuk konteks UMKM Indonesia**. Metodologi yang dibangun dirancang agar dapat di-transfer ke data lokal (KUR, fintech P2P, PNM Mekaar). Fokus utama: membangun model yang **akurat, fair, dan explainable** untuk mendukung inklusi keuangan.

## 1. Setup & Load Data dari Azure Blob Storage

In [ ]:
# Install dependencies (uncomment jika belum)
# !pip install pandas numpy matplotlib seaborn scikit-learn azure-storage-blob azure-ai-ml azure-identity

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visualisasi setting
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('✅ Libraries loaded')

In [ ]:
# OPSI A: Load dari Azure Blob (PRODUCTION-READY)
from azure.storage.blob import BlobServiceClient
from io import StringIO

# ⚠️ GANTI dengan connection string kamu (Azure Portal > Storage Account > Access keys)
CONNECTION_STRING = 'YOUR_CONNECTION_STRING_HERE'
CONTAINER_NAME = 'dataset'
BLOB_PATH = 'raw/credit_risk_dataset.csv'  # sesuaikan path

blob_service = BlobServiceClient.from_connection_string(CONNECTION_STRING)
blob_client = blob_service.get_blob_client(container=CONTAINER_NAME, blob=BLOB_PATH)
blob_data = blob_client.download_blob().readall()

df = pd.read_csv(StringIO(blob_data.decode('utf-8')))
print(f'✅ Dataset loaded from Azure Blob: {df.shape}')

In [ ]:
# OPSI B: Load via Azure ML Data Asset (NILAI BONUS untuk skor Azure!)
# Uncomment & jalankan kalau sudah register data asset di Azure ML Studio

# from azure.ai.ml import MLClient
# from azure.identity import DefaultAzureCredential
# import mltable

# ml_client = MLClient(
#     credential=DefaultAzureCredential(),
#     subscription_id='YOUR_SUBSCRIPTION_ID',
#     resource_group_name='rg-datathon-credit',
#     workspace_name='mlw-credit-risk-umkm'
# )
# data_asset = ml_client.data.get(name='credit_risk_raw', version='1')
# tbl = mltable.load(data_asset.path)
# df = tbl.to_pandas_dataframe()
# print(f'✅ Dataset loaded via Azure ML Data Asset: {df.shape}')

## 2. Data Quality Audit — First Look

**Tujuan:** Memahami struktur data sebelum analisis mendalam.

**Output yang juri suka:** Tabel ringkas yang menunjukkan kamu *paham* datanya, bukan asal lihat `.head()`.

In [ ]:
# Snapshot dataset
print('═' * 70)
print(f'📐 SHAPE: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('═' * 70)
df.head()

In [ ]:
# Audit table — comprehensive overview
audit = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'unique': df.nunique(),
    'sample': [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns]
})
audit

In [ ]:
# Statistik deskriptif numerik
df.describe().T.round(2)

In [ ]:
# Cek duplikasi
n_dup = df.duplicated().sum()
print(f'🔁 Duplicate rows: {n_dup:,} ({n_dup/len(df)*100:.2f}%)')

if n_dup > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'✅ Cleaned shape: {df.shape}')

## 3. Target Variable Analysis

**Kunci:** Apakah dataset imbalanced? Ini critical untuk pilihan metrik & strategi modeling nanti.

In [ ]:
# ⚠️ GANTI 'loan_status' dengan nama kolom target di dataset kamu
TARGET = 'loan_status'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribusi Target — Default vs Non-Default', fontweight='bold')
axes[0].set_xlabel('Loan Status (0=Non-Default, 1=Default)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
df[TARGET].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Proporsi Default Rate', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

default_rate = df[TARGET].mean() * 100
print(f'\n📊 INSIGHT: Default rate = {default_rate:.2f}%')
print(f'   Imbalance ratio = {(1-df[TARGET].mean())/df[TARGET].mean():.1f}:1')
print(f'\n💡 Implikasi:')
print(f'   - Dataset {"IMBALANCED — perlu strategi khusus" if default_rate < 30 else "BALANCED — straightforward"}')
print(f'   - Metrik utama: AUC-ROC, F1-Score, Precision-Recall (BUKAN accuracy!)')
print(f'   - Pertimbangkan: SMOTE, class_weight, atau threshold tuning')

## 4. Univariate Analysis — Distribusi Tiap Fitur

In [ ]:
# Pisahkan kolom numerik & kategorikal
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

if TARGET in num_cols:
    num_cols.remove(TARGET)

print(f'🔢 Numerik ({len(num_cols)}): {num_cols}')
print(f'🔤 Kategorikal ({len(cat_cols)}): {cat_cols}')

In [ ]:
# Distribusi fitur numerik
n = len(num_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4*nrows))
axes = axes.flatten() if n > 1 else [axes]

for i, col in enumerate(num_cols):
    df[col].hist(bins=40, ax=axes[i], edgecolor='black', alpha=0.7, color='#3498db')
    axes[i].set_title(f'Distribusi: {col}', fontweight='bold')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.1f}')
    axes[i].legend(fontsize=8)

for j in range(n, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur kategorikal
if cat_cols:
    n = len(cat_cols)
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4*nrows))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, col in enumerate(cat_cols):
        vc = df[col].value_counts().head(10)
        vc.plot(kind='barh', ax=axes[i], color='#9b59b6')
        axes[i].set_title(f'Top kategori: {col}', fontweight='bold')
        axes[i].invert_yaxis()
    
    for j in range(n, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

## 5. Outlier Detection — IQR Method

In [ ]:
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return ((series < lower) | (series > upper)).sum(), lower, upper

outlier_summary = []
for col in num_cols:
    n_out, low, up = detect_outliers_iqr(df[col])
    outlier_summary.append({
        'feature': col,
        'n_outliers': n_out,
        'pct_outliers': round(n_out / len(df) * 100, 2),
        'lower_bound': round(low, 2),
        'upper_bound': round(up, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('pct_outliers', ascending=False)
print('🚨 OUTLIER REPORT:')
outlier_df

In [ ]:
# Boxplot untuk visualisasi outlier
fig, axes = plt.subplots(1, len(num_cols), figsize=(4*len(num_cols), 5))
axes = [axes] if len(num_cols) == 1 else axes

for i, col in enumerate(num_cols):
    df.boxplot(column=col, ax=axes[i])
    axes[i].set_title(col, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Bivariate Analysis — Setiap Fitur vs Target

**Inilah jantung EDA yang juri tunggu.** Kita lihat fitur mana yang paling diskriminatif untuk prediksi default.

In [ ]:
# Numerik vs Target — pakai violin plot
fig, axes = plt.subplots((len(num_cols) + 2) // 3, 3, figsize=(15, 4*((len(num_cols) + 2) // 3)))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.violinplot(data=df, x=TARGET, y=col, ax=axes[i], palette=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'{col} vs Default', fontweight='bold')
    
    # Add mean comparison
    mean_0 = df[df[TARGET]==0][col].mean()
    mean_1 = df[df[TARGET]==1][col].mean()
    diff_pct = ((mean_1 - mean_0) / mean_0 * 100) if mean_0 != 0 else 0
    axes[i].text(0.5, 0.95, f'Δ Mean: {diff_pct:+.1f}%', 
                 transform=axes[i].transAxes, ha='center', 
                 fontsize=9, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

for j in range(len(num_cols), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Kategorikal vs Target — default rate per kategori
if cat_cols:
    fig, axes = plt.subplots((len(cat_cols) + 1) // 2, 2, figsize=(14, 4*((len(cat_cols) + 1) // 2)))
    axes = axes.flatten() if len(cat_cols) > 1 else [axes]
    
    for i, col in enumerate(cat_cols):
        default_by_cat = df.groupby(col)[TARGET].agg(['mean', 'count']).reset_index()
        default_by_cat = default_by_cat[default_by_cat['count'] >= 30]  # filter min sample
        default_by_cat['default_rate_pct'] = default_by_cat['mean'] * 100
        default_by_cat = default_by_cat.sort_values('default_rate_pct', ascending=False)
        
        bars = axes[i].barh(default_by_cat[col].astype(str), default_by_cat['default_rate_pct'], 
                             color=plt.cm.RdYlGn_r(default_by_cat['default_rate_pct']/default_by_cat['default_rate_pct'].max()))
        axes[i].axvline(df[TARGET].mean()*100, color='blue', linestyle='--', label=f'Avg: {df[TARGET].mean()*100:.1f}%')
        axes[i].set_title(f'Default Rate per {col}', fontweight='bold')
        axes[i].set_xlabel('Default Rate (%)')
        axes[i].legend()
        
        # Add count labels
        for j, (rate, cnt) in enumerate(zip(default_by_cat['default_rate_pct'], default_by_cat['count'])):
            axes[i].text(rate + 0.5, j, f'n={cnt}', va='center', fontsize=8)
    
    for j in range(len(cat_cols), len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

## 7. Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = df[num_cols + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Top correlations dengan target
target_corr = corr_matrix[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(f'\n🎯 TOP FITUR BERKORELASI dengan {TARGET}:')
print(target_corr.head(10).to_string())

## 8. 🛠️ Data Cleaning

**Strategi cleaning:**
- **Missing values:** Median untuk numerik, mode untuk kategorikal
- **Outlier ekstrem:** Cap di percentile 99 (winsorization)
- **Inconsistent values:** Standardize

In [ ]:
df_clean = df.copy()

# Missing values handling
for col in num_cols:
    if df_clean[col].isnull().any():
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f'   ✓ {col}: filled NaN dengan median ({median_val:.2f})')

for col in cat_cols:
    if df_clean[col].isnull().any():
        mode_val = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode_val)
        print(f'   ✓ {col}: filled NaN dengan mode ({mode_val})')

# Outlier capping (winsorization at 1% & 99%)
for col in num_cols:
    p1, p99 = df_clean[col].quantile([0.01, 0.99])
    n_capped = ((df_clean[col] < p1) | (df_clean[col] > p99)).sum()
    df_clean[col] = df_clean[col].clip(p1, p99)
    if n_capped > 0:
        print(f'   ✓ {col}: capped {n_capped} outliers ke [{p1:.2f}, {p99:.2f}]')

print(f'\n✅ Cleaning selesai. Shape: {df_clean.shape}')

## 9. 🚀 Feature Engineering — Domain-Driven Features

**Ini bagian yang akan membedakan kamu dari peserta lain.** Kita buat fitur dengan **logika bisnis kredit UMKM**, bukan asal kombinasi.

### Filosofi Feature Engineering Credit Scoring:
1. **Capacity** — kemampuan bayar (rasio pendapatan vs cicilan)
2. **Capital** — modal/aset yang dimiliki
3. **Conditions** — konteks ekonomi & pekerjaan
4. **Character** — riwayat kredit (jika ada)

In [ ]:
# ⚠️ SESUAIKAN nama kolom dengan dataset kamu yang sebenarnya
# Contoh asumsi kolom: person_age, person_income, person_emp_length, 
# loan_amnt, loan_int_rate, person_home_ownership, loan_intent, dll.

df_fe = df_clean.copy()

# === CAPACITY (kemampuan bayar) ===
if 'loan_amnt' in df_fe.columns and 'person_income' in df_fe.columns:
    df_fe['loan_to_income_ratio'] = df_fe['loan_amnt'] / (df_fe['person_income'] + 1)
    print('   ✓ loan_to_income_ratio: rasio pinjaman terhadap pendapatan')

if 'loan_int_rate' in df_fe.columns and 'loan_amnt' in df_fe.columns:
    df_fe['monthly_interest_burden'] = (df_fe['loan_amnt'] * df_fe['loan_int_rate'] / 100) / 12
    print('   ✓ monthly_interest_burden: beban bunga bulanan')

if 'monthly_interest_burden' in df_fe.columns and 'person_income' in df_fe.columns:
    df_fe['debt_service_ratio'] = df_fe['monthly_interest_burden'] / ((df_fe['person_income']/12) + 1)
    print('   ✓ debt_service_ratio: DSR — proxy beban bulanan')

# === CHARACTER (stabilitas) ===
if 'person_emp_length' in df_fe.columns:
    df_fe['emp_stability_score'] = pd.cut(df_fe['person_emp_length'], 
                                            bins=[-1, 1, 3, 5, 10, 100],
                                            labels=['very_new', 'new', 'medium', 'stable', 'very_stable'])
    print('   ✓ emp_stability_score: kategori stabilitas pekerjaan')

if 'person_age' in df_fe.columns:
    df_fe['age_group'] = pd.cut(df_fe['person_age'],
                                 bins=[0, 25, 35, 45, 60, 100],
                                 labels=['young', 'early_career', 'mid_career', 'senior', 'elderly'])
    print('   ✓ age_group: kelompok usia')

# === CAPITAL (struktur modal) ===
if 'person_income' in df_fe.columns:
    df_fe['income_log'] = np.log1p(df_fe['person_income'])
    df_fe['income_quartile'] = pd.qcut(df_fe['person_income'], q=4, 
                                        labels=['Q1_low', 'Q2_mid_low', 'Q3_mid_high', 'Q4_high'],
                                        duplicates='drop')
    print('   ✓ income_log + income_quartile')

# === CONDITIONS (konteks loan) ===
if 'loan_int_rate' in df_fe.columns:
    df_fe['interest_risk_band'] = pd.cut(df_fe['loan_int_rate'],
                                          bins=[0, 7, 12, 17, 100],
                                          labels=['low', 'medium', 'high', 'very_high'])
    print('   ✓ interest_risk_band: kategori suku bunga')

# === COMPOSITE: Risk Score Awal (intuitive baseline) ===
# Komposit sederhana berdasarkan domain knowledge
risk_signals = []
if 'loan_to_income_ratio' in df_fe.columns:
    risk_signals.append((df_fe['loan_to_income_ratio'] > 0.5).astype(int))
if 'loan_int_rate' in df_fe.columns:
    risk_signals.append((df_fe['loan_int_rate'] > 15).astype(int))
if 'person_emp_length' in df_fe.columns:
    risk_signals.append((df_fe['person_emp_length'] < 2).astype(int))

if risk_signals:
    df_fe['domain_risk_score'] = sum(risk_signals)
    print(f'   ✓ domain_risk_score: skor risiko intuitive (0-{len(risk_signals)})')

print(f'\n✅ Feature engineering selesai. Shape baru: {df_fe.shape}')
print(f'   Fitur baru ditambahkan: {df_fe.shape[1] - df_clean.shape[1]}')

In [ ]:
# Validasi: cek apakah fitur baru predictive terhadap target
new_features = [c for c in df_fe.columns if c not in df_clean.columns]
print(f'🔬 Validasi {len(new_features)} fitur baru:\n')

for feat in new_features:
    if df_fe[feat].dtype in ['int64', 'float64']:
        # Numerik: cek korelasi dengan target
        corr = df_fe[feat].corr(df_fe[TARGET])
        print(f'   {feat:30s}  →  corr dengan {TARGET}: {corr:+.3f}')
    else:
        # Kategorikal: cek default rate per kategori
        rates = df_fe.groupby(feat)[TARGET].mean()
        spread = rates.max() - rates.min()
        print(f'   {feat:30s}  →  spread default rate: {spread:.3f}')

## 10. 💾 Save Processed Dataset

Simpan dataset bersih ke folder `processed/` di Azure Blob untuk dipakai notebook modeling.

In [ ]:
# Save lokal dulu
OUTPUT_PATH = 'credit_risk_processed.csv'
df_fe.to_csv(OUTPUT_PATH, index=False)
print(f'✅ Saved lokal: {OUTPUT_PATH} ({df_fe.shape})')

# Upload ke Azure Blob — folder processed/
blob_client_out = blob_service.get_blob_client(
    container=CONTAINER_NAME, 
    blob='processed/credit_risk_processed.csv'
)
with open(OUTPUT_PATH, 'rb') as f:
    blob_client_out.upload_blob(f, overwrite=True)
print(f'✅ Uploaded ke Azure Blob: processed/credit_risk_processed.csv')

## 11. 📝 Ringkasan EDA — Insight Untuk Presentasi

**[ISI MANUAL berdasarkan hasil notebook kamu]**

### Temuan Kunci:
1. **Kualitas Data:** Dataset memiliki [N] baris dengan [X%] missing values yang sudah ditangani
2. **Imbalance:** Default rate [X%] — perlu strategi class weight / SMOTE
3. **Top Predictors (korelasi):** [list 5 fitur teratas]
4. **Domain Insight:** [contoh: "UMKM dengan loan-to-income ratio > 0.5 memiliki probabilitas default 2.3x lebih tinggi"]
5. **Fitur Baru:** [N] fitur engineered yang berbasis framework 4C (Capacity, Capital, Character, Conditions)

### Implikasi untuk Modeling (Notebook 02):
- Gunakan **AUC-ROC + F1** sebagai metrik utama
- Pertimbangkan **SMOTE** untuk handling imbalance
- Algoritma kandidat: **XGBoost, LightGBM, Random Forest** (handle non-linear & feature interaction)
- Wajib include **SHAP analysis** untuk explainability